# Tyler Deliverables — Solar Array Detector Results

Generates all three items from Tyler's May 6 email:
1. Error category test instances with annotations
2. mAP50 vs cumulative training size curve
3. Same test tiles re-run at multiple points on the curve

**Before running:** download `r3_cameron_20260509.pt` from Google Drive
(`solar-soiling/models/`) to your local `models/` folder.

All cells run on CPU — expect ~20-30 min total.


In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir('/home/cameron/repos/solar-soiling-ml')
sys.path.insert(0, 'src')
ENV = {**os.environ, 'PYTHONPATH': '.'}
OUT = Path('outputs/tyler_deliverable')
OUT.mkdir(parents=True, exist_ok=True)

MODELS = {
    'Baseline':       'models/sahi_baseline_train7.pt',
    'R1 v8 +50 Duke': 'models/r1_cameron_20260508.pt',
    'R3 v9 +150 Duke':'models/r3_cameron_20260509.pt',
}

missing = [f'{k}: {v}' for k, v in MODELS.items() if not Path(v).exists()]
if missing:
    print('MISSING weights -- download from Drive:')
    for m in missing:
        print(f'  {m}')
else:
    print('All weights present:')
    for k, v in MODELS.items():
        print(f'  {k}: {v}')


## Deliverable 2 — mAP50 vs. Training Size Curve

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# v9 label ramp results (2026-05-09)
ramp = [
    ('SAHI baseline', 174,  0, 0.563, '-',    '-',    'reference'),
    ('R0',            186,  0, 0.568, 0.642,  0.575,  'ok'),
    ('R1',            236,  50, 0.551, 0.609, 0.542,  'ok'),
    ('R2',            286, 100, 0.553, 0.639, 0.539,  'ok'),
    ('R3',            336, 150, 0.569, 0.632, 0.571,  'ok'),
    ('R4',            386, 200, 0.487, 0.582, 0.483,  'halt'),
    ('R5',            436, 250, 0.554, 0.701, 0.484,  'ok'),
]

fig, ax = plt.subplots(figsize=(10, 5))

x = [r[1] for r in ramp]
y = [r[3] for r in ramp]
colors = ['#e74c3c' if r[6]=='halt' else '#2ecc71' if r[6]=='reference' else '#3498db' for r in ramp]

ax.plot(x, y, color='#95a5a6', linewidth=1.5, zorder=1)
ax.scatter(x, y, c=colors, s=100, zorder=2)

# Annotate best
best_x, best_y = 336, 0.569
ax.annotate('R3 — production candidate\n(mAP50=0.569)', xy=(best_x, best_y),
            xytext=(best_x+30, best_y+0.02),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

# Baseline reference line
ax.axhline(0.563, color='green', linestyle='--', alpha=0.5, linewidth=1)
ax.text(180, 0.565, 'Baseline (0.563)', fontsize=8, color='green', alpha=0.7)

ax.set_xlabel('Cumulative training images (NAIP + Duke)', fontsize=11)
ax.set_ylabel('mAP50 (NAIP test set, 50 tiles)', fontsize=11)
ax.set_title('Solar Array Detector — Duke Data Ramp Curve (v9 labels)', fontsize=12)
ax.set_ylim(0.45, 0.62)
ax.grid(True, alpha=0.3)

legend_handles = [
    mpatches.Patch(color='#2ecc71', label='Baseline'),
    mpatches.Patch(color='#3498db', label='Clean'),
    mpatches.Patch(color='#e74c3c', label='HALT (regression > 0.07)'),
]
ax.legend(handles=legend_handles, fontsize=9)

out_path = OUT / 'ramp_curve.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')

# Print table
print()
print(f'{"Step":<18} {"Train imgs":>12} {"Duke":>8} {"mAP50":>8} {"P":>8} {"R":>8} {"Status":>10}')
print('-' * 76)
for name, n, duke, m50, p, r, status in ramp:
    p_s = f'{p:.3f}' if isinstance(p, float) else p
    r_s = f'{r:.3f}' if isinstance(r, float) else r
    print(f'{name:<18} {n:>12} {duke:>8} {m50:>8.3f} {p_s:>8} {r_s:>8} {status:>10}')


## Deliverable 1 — Error Category Overlays

Runs inference on the 50-tile test set with the best model (R3 v9),
categorises every prediction as TP/FP/FN, and renders overlay PNGs.
**~15-20 min on CPU.**


In [ ]:
# Run 05c on R3 (best model) — generates per_detection.csv
result = subprocess.run([
    sys.executable, 'scripts/05c_per_detection_rca.py',
    '--weights', 'models/r3_cameron_20260509.pt',
    '--data', 'data/yolo/naip/data.yaml',
    '--splits', 'test',
    '--conf', '0.05', '--iou', '0.50',
    '--sahi',
    '--run-name', 'R3_v9_tyler',
], env=ENV)
print('Done' if result.returncode == 0 else f'Failed (rc={result.returncode})')


In [ ]:
# Generate error category overlay PNGs
for bucket, args in [
    ('confident_fp',  ['--bucket', 'confident_fp', '--top', '20']),
    ('alone_tile_fp', ['--bucket-expr', 'class=fp AND num_other_panels_in_tile=0', '--top', '20', '--rank-by', 'conf', '--rank-desc']),
    ('worst_small_fn',['--bucket', 'worst_small_fn', '--top', '20']),
]:
    r = subprocess.run([
        sys.executable, 'scripts/labeling/18_bucket_overlays.py',
        '--csv', 'outputs/eval/R3_v9_tyler/per_detection.csv',
    ] + args, env=ENV)
    print(f'{bucket}: {"ok" if r.returncode == 0 else "failed"}')

# Copy to tyler deliverable folder
import shutil
viz_src = Path('outputs/label_viz/R3_v9_tyler')
viz_dst = OUT / 'error_categories'
if viz_dst.exists():
    shutil.rmtree(viz_dst)
if viz_src.exists():
    shutil.copytree(viz_src, viz_dst)
    for bucket in viz_dst.iterdir():
        pngs = list(bucket.glob('*.jpg')) + list(bucket.glob('*.png'))
        print(f'  {bucket.name}: {len(pngs)} tiles')


## Deliverable 3 — Same Tiles at Multiple Ramp Points

Picks the top confident-FP tiles from R3 and re-renders them for the
baseline and R1 so Tyler can see how individual predictions change
as data is added.


In [ ]:
# Run 05c on baseline and R1 if not already done
for name, weights, run_name in [
    ('Baseline', 'models/sahi_baseline_train7.pt', 'baseline_tyler'),
    ('R1 v8',   'models/r1_cameron_20260508.pt',  'R1_v8_tyler'),
]:
    csv_path = Path(f'outputs/eval/{run_name}/per_detection.csv')
    if csv_path.exists():
        print(f'{name}: already done -- skipping')
        continue
    print(f'Running 05c on {name}...')
    r = subprocess.run([
        sys.executable, 'scripts/05c_per_detection_rca.py',
        '--weights', weights,
        '--data', 'data/yolo/naip/data.yaml',
        '--splits', 'test',
        '--conf', '0.05', '--iou', '0.50',
    '--sahi',
        '--run-name', run_name,
    ], env=ENV)
    print(f'  Done' if r.returncode == 0 else f'  Failed')


In [ ]:
import pandas as pd, shutil

# Pick top 5 tiles from R3's confident_fp bucket
r3_csv = Path('outputs/eval/R3_v9_tyler/per_detection.csv')
df = pd.read_csv(r3_csv)
fp_df = df[(df['class'] == 'fp') & (df['conf'] >= 0.30)].copy()
top_tiles = fp_df.nlargest(5, 'conf')['tile_id'].unique()[:5]
print('Selected tiles for comparison:')
for t in top_tiles:
    print(f'  {t}')

# Render overlay for each tile across all three models
comparison_dir = OUT / 'tile_comparison'
comparison_dir.mkdir(exist_ok=True)

for run_name, label in [
    ('baseline_tyler', 'Baseline'),
    ('R1_v8_tyler',    'R1_v8_+50Duke'),
    ('R3_v9_tyler',    'R3_v9_+150Duke'),
]:
    viz_dir = Path(f'outputs/label_viz/{run_name}')
    for bucket in viz_dir.iterdir() if viz_dir.exists() else []:
        for tile_id in top_tiles:
            for img in list(bucket.glob(f'{tile_id}*')) + list(bucket.glob(f'*{tile_id[:12]}*')):
                dst = comparison_dir / f'{label}_{img.name}'
                shutil.copy(img, dst)

saved = list(comparison_dir.glob('*.jpg')) + list(comparison_dir.glob('*.png'))
print(f'Saved {len(saved)} comparison tiles to {comparison_dir}')


## Preview outputs inline

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Ramp curve
print('=== Ramp Curve ===')
display(Image(filename=str(OUT / 'ramp_curve.png'), width=700))

# Error categories -- show 2 from each bucket
print('\n=== Error Categories (R3 v9 best model) ===')
for bucket in sorted((OUT / 'error_categories').iterdir()) if (OUT / 'error_categories').exists() else []:
    imgs = sorted(bucket.glob('*.jpg')) + sorted(bucket.glob('*.png'))
    imgs = [i for i in imgs if 'manifest' not in i.name]
    if not imgs:
        continue
    print(f'\n-- {bucket.name} --')
    for img in imgs[:2]:
        display(Image(filename=str(img), width=600))

# Tile comparison
print('\n=== Tile Comparison Across Ramp Steps ===')
comparison_dir = OUT / 'tile_comparison'
imgs = sorted(comparison_dir.glob('*.jpg')) + sorted(comparison_dir.glob('*.png'))
for img in imgs[:9]:
    print(img.name)
    display(Image(filename=str(img), width=600))


## Output summary

All files saved to `outputs/tyler_deliverable/`:

```
tyler_deliverable/
  ramp_curve.png              -- Deliverable 2
  error_categories/
    confident_fp/             -- Deliverable 1: high-confidence FPs
    alone_tile_fp/            -- Deliverable 1: FPs on tiles we never labeled
    worst_small_fn/           -- Deliverable 1: missed small panels
  tile_comparison/            -- Deliverable 3: same tiles across Baseline / R1 / R3
```

**Red outline** = ground truth label. **Cyan outline** = model prediction (with confidence score).
